# 01 — Entendimento do Negócio e dos Dados
**Projeto:** Recomendador de jogos da Steam baseado no perfil recente do jogador
**Metodologia:** CRISP-DM — etapas 1 (Entendimento do Negócio) e 2 (Entendimento dos Dados)

## 1. Entendimento do Negócio
- **Problema:** a Steam tem dezenas de milhares de jogos; achar o próximo jogo parecido com o que a pessoa gosta é difícil.
- **Pergunta de pesquisa:** usar vários jogos recentes (5) gera recomendações mais coerentes do que usar apenas um jogo?
- **Mudança em relação ao seminário (feedback do professor):** incorporamos uma segunda base, com **interações reais de usuários**.
  Isso permite (a) montar perfis reais de jogadores, (b) medir a qualidade das recomendações com métricas objetivas e
  (c) comparar modelos de conteúdo com modelos colaborativos.

| Base | Fonte | Papel no projeto |
|---|---|---|
| A — Steam Games Dataset | Kaggle (Hubert Sidorowicz) | metadados: gêneros, categorias, tags, avaliações |
| B — Game Recommendations on Steam | Kaggle (Anton Kozyriev) | interações usuário × jogo (recomendou? horas? data) |

In [ ]:
import sys
from pathlib import Path

RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(RAIZ))

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src import config, carregamento as c, preparacao as p, graficos as g

config.criar_pastas()
g.aplicar_estilo()
pd.set_option("display.max_columns", 60)
pd.set_option("display.float_format", lambda v: f"{v:,.4f}")
print("Percentual de usuários amostrados:", config.PERCENTUAL_USUARIOS, "%")

## 2. Base A — metadados dos jogos

In [ ]:
A = c.carregar_base_a()
print(f"Base A: {A.shape[0]:,} linhas x {A.shape[1]} colunas | {c.memoria_mb(A):,.0f} MB em memória")
A.head()

In [ ]:
faltantes_a = c.resumo_colunas(A)
faltantes_a.to_csv(config.RESULTADOS / "tabela_colunas_base_a.csv")
faltantes_a

In [ ]:
problemas_a = {
    "linhas": len(A),
    "app_id_duplicados": int(A["app_id"].duplicated().sum()),
    "nome_vazio": int(A["name"].isna().sum()),
    "sem_genero": int(A["genres"].map(p.parse_lista).map(len).eq(0).sum()),
    "sem_tags": int(A["tags"].map(p.parse_lista).map(len).eq(0).sum()) if "tags" in A else None,
    "sem_nenhuma_avaliacao": int(((A["positive"].fillna(0) + A["negative"].fillna(0)) == 0).sum()),
}
problemas_a

### Gêneros mais frequentes (figura usada no seminário, agora reproduzível em código)

In [ ]:
generos = (A.drop_duplicates("app_id")["genres"].map(p.parse_lista)
             .explode().dropna().value_counts().head(10))
fig, ax = plt.subplots(figsize=(8, 4.5))
sns.barplot(x=generos.values, y=generos.index, color=g.AZUL, width=0.7, saturation=1, ax=ax)
ax.set_ylabel("")
g.rotular_barras(ax, horizontal=True)
ax.set_title("Figura 1 — Os 10 gêneros mais frequentes (base A)")
ax.set_xlabel("Quantidade de jogos")
g.eixo_br(ax, "x")
ax.grid(axis="y", visible=False)
g.salvar(fig, "fig01_generos_top10")
plt.show()

### Volume de avaliações por jogo — justificativa para um volume mínimo no filtro de qualidade

In [ ]:
total_av = (A["positive"].fillna(0) + A["negative"].fillna(0))
print(total_av.describe(percentiles=[.25, .5, .75, .9, .99]))
print(f"Jogos com menos de {config.MIN_AVALIACOES_QUALIDADE} avaliações: {(total_av < config.MIN_AVALIACOES_QUALIDADE).mean():.1%}")

fig, ax = plt.subplots()
sns.histplot(np.log10(total_av + 1), bins=50, color=g.AZUL, edgecolor="white", linewidth=0.5, alpha=1, ax=ax)
ax.axvline(np.log10(config.MIN_AVALIACOES_QUALIDADE + 1), color=g.TEXTO_SECUNDARIO, ls="--", lw=1)
ax.text(np.log10(config.MIN_AVALIACOES_QUALIDADE + 1) + 0.05, ax.get_ylim()[1] * 0.9,
        f"mínimo = {config.MIN_AVALIACOES_QUALIDADE}", color=g.TEXTO_SECUNDARIO)
ax.set_title("Figura 2 — Distribuição do total de avaliações por jogo (escala log)")
ax.set_xlabel("log10(total de avaliações + 1)")
ax.set_ylabel("Quantidade de jogos")
g.salvar(fig, "fig02_distribuicao_avaliacoes")
plt.show()

### Faixas de taxa negativa (regra de negócio apresentada no seminário)

In [ ]:
taxa = np.where(total_av > 0, A["negative"].fillna(0) / total_av.replace(0, np.nan), np.nan)
faixas = pd.Series([p.classificar_taxa_negativa(t, n) for t, n in zip(np.nan_to_num(taxa, nan=1), total_av)])
ordem = ["Extremamente positivas", "Muito positivas", "Positivas",
         "Ligeiramente positivas / mistas", "Negativas (descartadas)", "Poucas avaliações"]
contagem_faixas = faixas.value_counts().reindex(ordem, fill_value=0)

fig, ax = plt.subplots(figsize=(8, 4))
sns.barplot(x=contagem_faixas.values, y=contagem_faixas.index, color=g.AZUL, width=0.7, saturation=1, ax=ax)
ax.set_ylabel("")
g.rotular_barras(ax, horizontal=True)
ax.set_title("Figura 3 — Jogos por faixa de taxa negativa (base A)")
ax.set_xlabel("Quantidade de jogos")
g.eixo_br(ax, "x")
ax.grid(axis="y", visible=False)
g.salvar(fig, "fig03_faixas_taxa_negativa")
plt.show()
contagem_faixas

## 3. Base B — interações de usuários

In [ ]:
gb = c.carregar_b_games()
users = c.carregar_b_users()
meta = c.carregar_b_metadata()
recs, resumo_recs = c.carregar_recomendacoes()

print(f"games.csv:          {gb.shape}")
print(f"users.csv:          {users.shape}")
print(f"games_metadata:     {meta.shape}")
print(f"recommendations:    {resumo_recs['linhas_base_completa']:,} linhas na base completa")
print(f"amostra ({config.PERCENTUAL_USUARIOS}% usuários): {len(recs):,} linhas | {c.memoria_mb(recs):,.0f} MB")
recs.head()

In [ ]:
tab_b = pd.concat({"games.csv": c.resumo_colunas(gb), "recommendations.csv": c.resumo_colunas(recs)})
tab_b.to_csv(config.RESULTADOS / "tabela_colunas_base_b.csv")
tab_b

### Desbalanceamento da variável `is_recommended`
A grande maioria das avaliações é positiva. Isso orienta a decisão de modelagem: tratamos o problema como
**feedback implícito positivo** (o usuário recomendou o jogo = gostou), e não como classificação binária,
onde a classe negativa seria minoritária e pouco informativa para recomendar.

In [ ]:
pos = resumo_recs["positivas_base_completa"]
neg = resumo_recs["linhas_base_completa"] - pos
bal = pd.Series({"Recomendou (positiva)": pos, "Não recomendou (negativa)": neg})

fig, ax = plt.subplots(figsize=(7, 3.2))
sns.barplot(x=bal.values, y=bal.index, color=g.AZUL, width=0.6, saturation=1, ax=ax)
ax.set_ylabel("")
for i, v in enumerate(bal.values):
    ax.text(v, i, f"  {g.br(v)} ({g.br(v / bal.sum() * 100, '{:.1f}')}%)", va="center", color=g.TEXTO_SECUNDARIO)
ax.margins(x=0.35)
ax.set_title("Figura 4 — Balanceamento de is_recommended (base B completa)")
ax.set_xlabel("Quantidade de avaliações")
g.eixo_br(ax, "x")
ax.grid(axis="y", visible=False)
g.salvar(fig, "fig04_balanceamento_is_recommended")
plt.show()

### Interações por usuário — a maioria avalia poucos jogos (cauda longa)

In [ ]:
por_usuario = recs[recs["is_recommended"]].groupby("user_id").size()
print(por_usuario.describe(percentiles=[.5, .75, .9, .99]))
print(f"Usuários com >= {config.MIN_INTERACOES_USUARIO} jogos recomendados: "
      f"{(por_usuario >= config.MIN_INTERACOES_USUARIO).mean():.1%}")

fig, ax = plt.subplots()
sns.histplot(por_usuario.clip(upper=50), discrete=True, color=g.AZUL, edgecolor="white", linewidth=0.5, alpha=1, ax=ax)
ax.axvline(config.MIN_INTERACOES_USUARIO, color=g.TEXTO_SECUNDARIO, ls="--", lw=1)
ax.set_yscale("log")
ax.set_title("Figura 5 — Jogos recomendados por usuário (valores acima de 50 agrupados)")
ax.set_xlabel("Jogos recomendados pelo usuário")
ax.set_ylabel("Usuários (escala log)")
g.salvar(fig, "fig05_interacoes_por_usuario")
plt.show()

### Concentração da popularidade — poucos jogos recebem a maior parte das interações

In [ ]:
por_jogo = recs[recs["is_recommended"]].groupby("app_id").size().sort_values(ascending=False)
x = np.arange(1, len(por_jogo) + 1) / len(por_jogo) * 100
y = por_jogo.cumsum().to_numpy() / por_jogo.sum() * 100
top10 = y[int(len(y) * 0.10) - 1]

fig, ax = plt.subplots()
ax.plot(x, y, color=g.AZUL)
ax.plot([0, 100], [0, 100], color=g.GRADE, lw=1, ls="--")
ax.scatter([10], [top10], color=g.AZUL, s=40, zorder=3, edgecolor="white", linewidth=1.5)
ax.annotate(f"10% dos jogos = {top10:.0f}% das interações", (10, top10), xytext=(18, top10 - 12),
            color=g.TEXTO, arrowprops=dict(arrowstyle="-", color=g.TEXTO_SECUNDARIO, lw=0.8))
ax.set_title("Figura 6 — Curva de concentração das interações por jogo")
ax.set_xlabel("% dos jogos (do mais para o menos popular)")
ax.set_ylabel("% acumulado das interações")
g.salvar(fig, "fig06_concentracao_popularidade")
plt.show()

## 4. Junção das bases A e B

In [ ]:
cobertura = gb["app_id"].isin(A["app_id"]).mean()
tags_a = A["tags"].map(p.parse_lista).map(len) if "tags" in A else pd.Series([0])
tags_b = meta["tags"].map(p.parse_lista).map(len)
print(f"Jogos da base B encontrados na base A: {cobertura:.1%}")
print(f"Média de tags por jogo — base A: {tags_a.mean():.1f} | base B: {tags_b.mean():.1f}")

In [ ]:
resumo_entendimento = {
    "base_a_linhas": int(len(A)),
    "base_a_colunas_usadas": int(A.shape[1]),
    **{f"base_a_{k}": v for k, v in problemas_a.items()},
    "base_b_jogos": int(len(gb)),
    "base_b_usuarios": int(len(users)),
    **{f"base_b_{k}": v for k, v in resumo_recs.items()},
    "cobertura_b_em_a": float(cobertura),
    "top10pct_jogos_share_interacoes": float(top10),
    "mediana_jogos_por_usuario_amostra": float(por_usuario.median()),
}
with open(config.RESULTADOS / "resumo_entendimento.json", "w", encoding="utf-8") as f:
    json.dump(resumo_entendimento, f, ensure_ascii=False, indent=2, default=float)
resumo_entendimento

## Principais achados desta etapa (execução registrada neste repositório)
- **Formato misto na base A:** gêneros e categorias aparecem ora como texto simples, ora como objetos da API da Steam
  (`{"id": "1", "description": "Action"}`) → o leitor `parse_lista` trata os dois formatos.
- **Faltantes na base A:** cerca de 8,4 mil jogos sem gênero, 61,5 mil sem tags e 61,9 mil sem nenhuma avaliação.
- **Avaliações muito assimétricas:** mediana de 3 avaliações por jogo e 78,8% dos jogos com menos de 50 →
  a taxa negativa do seminário precisa de volume mínimo.
- **Desbalanceamento (B):** 85,8% das 41,2 milhões de avaliações são positivas → modelagem como feedback implícito positivo.
- **Cauda longa:** mediana de 1 jogo recomendado por usuário, só 6,2% têm 8 ou mais, e 10% dos jogos concentram 91% das
  interações → baseline de popularidade obrigatório e cobertura do catálogo como métrica.
- **Junção:** 77,5% dos jogos da base B estão na base A; para os demais, as tags da base B descrevem o jogo.